> MiniLM

Nah ini seru nih, kita sering banget laa yaa nemu model ini tuu dipake klo emang ada constraint di computing time or memory gitu. Kenapa kok bisa worth? oke kita bisa bahas dl gimana si paradigma dari MiniLM itu sendiri, klo kita ngeliat model knowledge distillation lain let say DistilBERT -> simplenya dia niru output logits (hasil akhir) dari si gurunya(big model). MiniLM disini beda, instead of ngajarin ke murid "cuma" hasil akhir mending kita ajarin "how mikirnya". Btw, keluarga transformer itu "berpikir" dengan self-att itu, so MiniLM maksa student utk niruin attention maps (hubungan antar token) dari si guru.

Hal yg perlu kita ingat juga bro matriks attention itu kan perkaliannya as simple $TxT$ (token_len*token_len), regardlesss of berapapun dimensi embeddingnya ($d_{model}$) artinya apa? yaudah si guru dengan dimensi embedding yg gede banget 768 gitu bisa banget ngajarin murid dengan $d_{model}$ = 256 scara lsg gitu tanpa transform dimension yg aneh2. Nah gasken sekarang kita sikat implementasinya yg focus di Distillation Loss

In [ ]:
"""
Objective: 
Model Student (kecil) belajar meniru Attention Weights dari Model Teacher (besar).

Key Property:
Karena Attention Weight matrix selalu berukuran (B, H, T, T), kita bisa langsung
pakai KL-Divergence untuk mengukur perbedaan distribusi probabilitas attention
antara Teacher dan Student, walaupun d_model-nya beda jauh!
=============================================================================
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


In [2]:
# dummy encoder, disini kita ngereturn attention weights dari smua layer
class SimpleSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv = nn.Linear(d_model,d_model*3)
        self.out = nn.Linear(d_model,d_model)
        self.scale = self.head_dim ** -0.5
    
    def forward(self,x):
        B, T, _ = x.shape
        # proyeksi Q,K,V
        qkv = self.qkv(x).reshape(B,T,3,self.n_heads,self.head_dim)
        qkv = qkv.permute(2,0,3,1,4) # ini kan jadi dimensinya -> (3,B,H,T,hd)
        q,k,v = qkv[0],qkv[1],qkv[2]

        # hitung attention scores
        scores = (q @ k.transpose(-2,-1)) * self.scale # (B,H,T,T)
        attn_weights = F.softmax(scores,dim=-1)  # (B,H,T,T)

        # weighted sum of values
        context = (attn_weights @ v).transpose(1,2).reshape(B,T,-1)
        out = self.out(context)
        return out, attn_weights
